# 05 — Thiên lệch nguồn và độ tin cậy dữ liệu

**Phụ trách:** Tiết Trọng Khiêm  
**Trạng thái:** Đang kiểm định nguồn dữ liệu, phân bố nhãn và nguy cơ rò rỉ giữa các tập.  
**Đầu ra:** Bảng source × label × split, thống kê domain/provenance, biểu đồ EDA và nhận xét giới hạn dữ liệu; không xuất CSV/ảnh riêng.

Notebook phục vụ **K5.1 — Source-label audit** và một số kiểm tra hỗ trợ **K5.4 — Data reliability gate**; không huấn luyện mô hình hay gán nhãn URL mới. **K5.2 — External evaluation protocol** và blocker K5.3 được trình bày trong docs/external_datasheet.md và reports/week5_data_reliability.md.

Trong kế hoạch thí nghiệm, **E1** dùng URL features truyền thống để đo baseline; **E2** dùng URL features cùng Brand Features để đo cải thiện khi thêm Brand Features. Phạm vi Core gold/silver train/validation bên dưới là audit dữ liệu, không phải định nghĩa E1/E2. Official Test được giữ cho đánh giá cuối.


## 0. Môi trường và đường dẫn

Mặc định chạy local: cài `requirements.txt` trong `.venv`, mở notebook bằng VS Code/Jupyter và chọn kernel của `.venv`. Dataset ở `data/` hoặc vị trí `PHISHGUARD_DATA_DIR` trên máy mình. Ô dưới chỉ nạp đường dẫn/config; Colab là lựa chọn thêm theo README.

In [ ]:
from pathlib import Path
import os
import sys
import tomllib

start = Path(os.environ.get("PHISHGUARD_ROOT", Path.cwd())).expanduser().resolve()
candidates = (start, *start.parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / "pyproject.toml").is_file()
     and (p / "src" / "phishguard_vn").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Chuyen working directory vao repo hoac dat PHISHGUARD_ROOT "
        "toi thu muc du an tren may ban."
    )
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from phishguard_vn.paths import get_data_dir, get_config_path
DATA_DIR = get_data_dir()
with get_config_path().open("rb") as handle:
    CONFIG = tomllib.load(handle)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Config:", CONFIG["experiment"]["name"])

## 1. Nạp dữ liệu và kiểm tra cấu trúc

Dùng bảng URL chính `dataset_url.csv`. Giữ nguyên dữ liệu gốc; các bảng và biểu đồ kiểm toán sẽ hiển thị trực tiếp trong notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = DATA_DIR / "dataset_url.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)
audit_preview = df.loc[
    df["tier"].isin({"gold", "silver"})
    & df["split"].isin(["train", "val", "validation"])
]
print("Đã nạp dataset; preview chỉ hiển thị Core gold/silver train/validation.")
print("Số cột:", df.shape[1])
display(audit_preview.head())


In [ ]:
required_cols = ["id", "url", "url_norm", "domain", "source", "label", "split"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset thiếu cột bắt buộc: {missing_cols}")

print("Đã kiểm tra các cột bắt buộc.")

## 2. Phân bố nhãn, nguồn và split

Trước tiên kiểm tra quy mô từng lớp và từng nguồn. Bảng này giúp phát hiện nguồn chỉ chứa một nhãn — một dấu hiệu quan trọng của source-label confounding.

In [ ]:
# Audit chỉ dùng Core eligible gold/silver trong train/validation; không đọc Official Test.
CORE_TIERS = {"gold", "silver"}
eda_df = df[
    df["tier"].isin(CORE_TIERS)
    & df["split"].isin(["train", "val", "validation"])
].copy()
print("Số dòng Core eligible train/validation:", len(eda_df))
display(eda_df["label"].value_counts(dropna=False).rename("so_ban_ghi").to_frame())
display(eda_df["source"].value_counts(dropna=False).rename("so_ban_ghi").to_frame())
display(eda_df["split"].value_counts(dropna=False).rename("so_ban_ghi").to_frame())


**Phạm vi thống kê:** Các bảng dưới đây chỉ dùng Core gold/silver trong train/validation. Không hiển thị phân bố nhãn, nguồn hoặc split của Official Test.


## 3. Bảng source × label × split (train/validation)

Đếm bản ghi theo nguồn, nhãn và split trong Core eligible để nhận diện source-label confounding.


In [ ]:
source_label_split = pd.crosstab(
    [eda_df["source"], eda_df["split"]], eda_df["label"]
)
display(source_label_split)


In [ ]:
source_label = pd.crosstab(eda_df["source"], eda_df["label"])
source_label_rate = source_label.div(source_label.sum(axis=1), axis=0)
display(source_label_rate.round(3))


**Nhận xét trên Core eligible:** Cả 5 nguồn chỉ chứa một nhãn: tinnhiemmang có 2.100 phishing; tranco 5.105, tinnhiem_web 3.633, tranco_vn 2.159 và tinnhiem_org 1.668 đều benign. Nguồn vì vậy xác định hoàn toàn nhãn trong tập này; đây là source-label confounding nghiêm trọng khi diễn giải E1/E2.


## 4. Cơ cấu nguồn trong train và validation

Bảng chỉ mô tả train/validation thuộc Core eligible. Official Test được giữ cho đánh giá cuối.


In [ ]:
source_split = pd.crosstab(eda_df["source"], eda_df["split"])
source_split_rate = source_split.div(source_split.sum(axis=0), axis=1)
display(source_split)
display(source_split_rate.round(3))


**Cách diễn giải:** So sánh cơ cấu nguồn giữa train và validation; do mỗi nguồn trong Core chỉ có một nhãn, thay đổi cơ cấu nguồn đồng thời làm thay đổi cơ cấu nhãn. Không dùng kết quả này để ra quyết định dựa trên Official Test.


## 5. Trực quan hóa phân bố nguồn và nhãn

Biểu đồ là phần hỗ trợ đọc kết quả, không thay thế các bảng số liệu.

In [ ]:
source_label.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Core train/validation: số bản ghi theo nguồn và nhãn")
plt.xlabel("Nguồn")
plt.ylabel("Số bản ghi")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
source_split_rate.T.plot(kind="bar", stacked=True, figsize=(8, 5))
plt.title("Core train/validation: tỷ lệ nguồn theo split")
plt.xlabel("Split")
plt.ylabel("Tỷ lệ")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Kiểm tra trùng lặp và nhãn mâu thuẫn

Kiểm tra ID, URL và URL chuẩn hóa trong Core eligible train/validation.


In [ ]:
duplicate_summary = pd.DataFrame({
    "Trường": ["id", "url", "url_norm"],
    "Số dòng thuộc nhóm trùng": [
        eda_df.duplicated("id", keep=False).sum(),
        eda_df.duplicated("url", keep=False).sum(),
        eda_df.duplicated("url_norm", keep=False).sum(),
    ],
})
display(duplicate_summary)


In [ ]:
label_conflicts = {}
for col in ["id", "url", "url_norm"]:
    counts = eda_df.groupby(col, dropna=False)["label"].nunique()
    label_conflicts[col] = int((counts > 1).sum())
display(pd.Series(label_conflicts, name="Nhóm có nhiều nhãn"))


**Nhận xét:** Kết quả trùng lặp và nhãn mâu thuẫn bên trên chỉ áp dụng cho Core eligible train/validation.


## 7. Độ đa dạng domain theo nguồn

Thống kê trên Core eligible train/validation; domain được kiểm tra thêm theo PSL ở phần sau.


In [ ]:
domain_summary = eda_df.groupby("source").agg(
    so_ban_ghi=("id", "size"),
    url_duy_nhat=("url_norm", "nunique"),
    domain_duy_nhat=("domain", "nunique"),
)
domain_summary["domain_tren_ban_ghi"] = domain_summary["domain_duy_nhat"] / domain_summary["so_ban_ghi"]
display(domain_summary.sort_values("so_ban_ghi", ascending=False))


In [ ]:
domain_frequency = eda_df.groupby(["source", "domain"]).size().rename("so_ban_ghi")
display(domain_frequency.sort_values(ascending=False).head(15).to_frame())


**Nhận xét:** Đây là thống kê mô tả domain trong Core train/validation; không dùng phân bố Official Test để kết luận chất lượng mô hình.


## 8. Overlap hostname và registrable domain giữa train/validation

Parser urllib.parse + ipaddress + tldextract với PSL offline được áp dụng trên Core eligible.


In [ ]:
import ipaddress
import re
import tldextract
from urllib.parse import urlsplit

if tldextract.__version__ != "5.4.0":
    raise RuntimeError(f"Cần tldextract==5.4.0; đang dùng {tldextract.__version__}")
psl_extract = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=False)

def parse_for_audit(raw_url):
    if pd.isna(raw_url) or not str(raw_url).strip():
        return {"hostname": None, "registrable_domain": None, "parse_error": "missing_url"}
    value = str(raw_url).strip()
    candidate = value if value.startswith("//") or re.match(r"^[A-Za-z][A-Za-z0-9+.-]*://", value) else "http://" + value
    if candidate.startswith("//"):
        candidate = "http:" + candidate
    try:
        parsed = urlsplit(candidate)
        _ = parsed.port
        host = parsed.hostname
    except ValueError as exc:
        reason = "invalid_port" if "port" in str(exc).lower() else "parsing_error"
        return {"hostname": None, "registrable_domain": None, "parse_error": reason}
    if not host:
        return {"hostname": None, "registrable_domain": None, "parse_error": "missing_hostname"}
    host = host.rstrip(".").lower()
    try:
        ipaddress.ip_address(host)
        return {"hostname": host, "registrable_domain": None, "parse_error": "ip_no_registrable_domain"}
    except ValueError:
        pass
    try:
        host = host.encode("idna").decode("ascii")
    except UnicodeError:
        return {"hostname": host, "registrable_domain": None, "parse_error": "invalid_idn"}
    ext = psl_extract(host)
    registrable = ext.top_domain_under_public_suffix or None
    if registrable is None:
        return {"hostname": host, "registrable_domain": None, "parse_error": "unresolved_domain"}
    return {"hostname": host, "registrable_domain": registrable, "parse_error": None}

parsed = pd.DataFrame(eda_df["url"].map(parse_for_audit).tolist(), index=eda_df.index)
core_df = pd.concat([eda_df.copy(), parsed.add_prefix("parsed_")], axis=1)
core_df["stored_hostname"] = core_df["domain"].fillna("").astype(str).str.strip().str.lower().str.rstrip(".")
core_df["hostname_mismatch"] = (
    core_df["parsed_hostname"].notna()
    & core_df["stored_hostname"].ne("")
    & core_df["parsed_hostname"].ne(core_df["stored_hostname"])
)


**Chính sách parser:** tldextract chạy offline với PSL đóng gói trong package, không tính private suffix. IP hợp lệ không có registrable domain. Không phát hiện overlap tên miền không loại trừ mọi campaign/shared-infrastructure leakage.


## 9. Overlap và trường hợp bị loại khỏi phân tích domain

So sánh hostname và registrable domain giữa train/validation; liệt kê parser error và mismatch để rà soát.


In [ ]:
def cross_split_values(column):
    usable = core_df[core_df[column].notna() & core_df["split"].isin(["train", "val", "validation"])]
    counts = usable.groupby(column)["split"].nunique()
    return set(counts[counts > 1].index)

host_overlap = cross_split_values("parsed_hostname")
registrable_overlap = cross_split_values("parsed_registrable_domain")
parse_errors = core_df[core_df["parsed_parse_error"].notna()]
mismatches = core_df[core_df["hostname_mismatch"]]
print("Hostname overlap:", len(host_overlap), "| registrable-domain overlap:", len(registrable_overlap))
print("Parser unresolved/error:", len(parse_errors), "| stored hostname mismatch:", len(mismatches))
display(parse_errors[["id", "url", "split", "label", "parsed_parse_error"]])
display(mismatches[["id", "url", "domain", "parsed_hostname", "split"]])


**Ghi chú:** Các URL parser lỗi/unresolved và mismatch được liệt kê để loại khỏi kết luận domain-level, không âm thầm gán registrable domain.


## 10. Provenance và thời gian thu thập

Thống kê metadata chỉ trên Core eligible train/validation. collected_at là thời điểm sự kiện theo nguồn; scraped_at là thời điểm nhóm thu thập.


In [ ]:
provenance_cols = [c for c in ["label_source", "tier", "status"] if c in eda_df.columns]
for col in provenance_cols:
    print(f"Phân bố {col} (Core eligible)")
    display(eda_df[col].value_counts(dropna=False).to_frame("count"))


**Phạm vi:** Các bảng metadata ở trên chỉ mô tả Core eligible, không bao gồm Official Test.


In [ ]:
if "collected_at" in eda_df.columns:
    date_audit = eda_df.groupby(["source", "label"], dropna=False).agg(
        total_records=("id", "size"),
        nonempty_dates=("collected_at", lambda s: s.notna().sum()),
        missing_dates=("collected_at", lambda s: s.isna().sum()),
    )
    display(date_audit)


In [ ]:
if "collected_at" in eda_df.columns:
    dates_raw = eda_df["collected_at"]
    dates = pd.to_datetime(dates_raw, errors="coerce", format="mixed", dayfirst=True)
    print("Core collected_at: nonempty", int(dates_raw.notna().sum()),
          "| parsed", int(dates.notna().sum()),
          "| nonempty parse failures", int((dates_raw.notna() & dates.isna()).sum()),
          "| truly missing", int(dates_raw.isna().sum()))
if "scraped_at" in eda_df.columns:
    scraped = pd.to_datetime(eda_df["scraped_at"], errors="coerce", format="mixed", utc=True)
    print("Core scraped_at: nonempty", int(eda_df["scraped_at"].notna().sum()),
          "| parsed", int(scraped.notna().sum()))


**Ngày:** format="mixed" nhận diện các định dạng ngày khác nhau; ô trống được thống kê riêng, không tính là lỗi parse. Trên toàn dataset có 21.661 collected_at không rỗng và parse được; notebook chỉ hiển thị thống kê riêng cho Core eligible.


## 11. Tóm tắt reliability trong Core eligible

Bảng tổng hợp cấu trúc, overlap hostname/registrable domain và parser errors trong train/validation.


In [ ]:
def cross_split_overlap_count(column):
    usable = core_df[core_df[column].notna() & core_df["split"].isin(["train", "val", "validation"])]
    split_counts = usable.groupby(column)["split"].nunique()
    return int((split_counts > 1).sum())

reliability = pd.DataFrame([
    {"Kiểm tra": "Core eligible rows", "Số lượng": len(core_df)},
    {"Kiểm tra": "ID duplicate rows", "Số lượng": int(core_df.duplicated("id", keep=False).sum())},
    {"Kiểm tra": "URL duplicate rows", "Số lượng": int(core_df.duplicated("url", keep=False).sum())},
    {"Kiểm tra": "url_norm duplicate rows", "Số lượng": int(core_df.duplicated("url_norm", keep=False).sum())},
    {"Kiểm tra": "hostname overlap train/validation", "Số lượng": cross_split_overlap_count("parsed_hostname")},
    {"Kiểm tra": "registrable-domain overlap train/validation", "Số lượng": cross_split_overlap_count("parsed_registrable_domain")},
    {"Kiểm tra": "stored hostname mismatch", "Số lượng": int(core_df["hostname_mismatch"].sum())},
    {"Kiểm tra": "parser unresolved/error", "Số lượng": int(core_df["parsed_parse_error"].notna().sum())},
])
display(reliability)


## 12. Kết luận kiểm toán

- **Phạm vi:** K5.1/K5.4 audit dùng Core gold/silver trong train/validation; Official Test được giữ cho đánh giá cuối, không dùng phân bố Test để chọn mô hình/threshold.
- **Source bias:** 5 nguồn Core đều chỉ có một nhãn, nên source-label confounding hoàn toàn trong phạm vi audit; cần báo cáo metric theo nguồn khi so sánh E1 (URL features truyền thống) và E2 (URL + Brand Features).
- **Reliability:** Đối chiếu hostname và registrable domain theo PSL; xem số liệu reliability, parser errors và mismatch ở trên. Kết quả không loại trừ mọi dạng campaign leakage.
- **Ngày:** Mixed-format được parse riêng với ô thiếu; 21.661 giá trị collected_at không rỗng trên toàn dataset parse được, còn ngày thiếu là thiếu thật.
- **K5.2/K5.3:** Protocol external, schema, tiêu chí nguồn/nhãn và blocker chưa có pilot dataset được ghi trong docs/external_datasheet.md và reports/week5_data_reliability.md.
